# RF Tuned Current Stress

Template eksperimen Current Stress dengan tracking MLflow yang konsisten.

In [ ]:
from pathlib import Path
import tempfile
import mlflow
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from mlflow_utils import (
    DATASET_NAME,
    DATASET_VERSION,
    EXPERIMENT_NAME,
    RANDOM_STATE,
    TEST_SIZE,
    build_preprocessor,
    evaluate_classification,
    load_dataset,
    log_and_register_model,
    log_classification_artifacts,
    log_dataset_inputs,
    log_run_metadata,
    select_features,
    set_seeds,
)


In [ ]:
# 1) Header & Config
set_seeds(RANDOM_STATE)

repo_root = Path.cwd().resolve()
for parent in [repo_root, *repo_root.parents]:
    if (parent / ".git").exists():
        repo_root = parent
        break

tracking_dir = repo_root / "mlruns"
tracking_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"file:{tracking_dir.resolve().as_posix()}")
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

# 2) Load dataset + feature set
_, X, y = load_dataset()
X = select_features(X, feature_group="all")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# 3) Build preprocessing transformer
num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
preprocessor = build_preprocessor(num_cols, cat_cols)
base_pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))])

# 4) Train via GridSearchCV
param_grid = {
    "classifier__n_estimators": [200, 300],
    "classifier__max_depth": [None, 8, 12],
    "classifier__min_samples_split": [2, 4],
    "classifier__class_weight": [None, "balanced"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(base_pipeline, param_grid=param_grid, cv=cv, scoring="f1_weighted", n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)
metrics = evaluate_classification(y_test, y_pred, y_proba)

# 5) MLflow logging
run_name = "RF Tuned - All Features"
description = f"RF tuned via GridSearchCV; features=all; dataset=current_stress_v1; best_params={search.best_params_}"
with mlflow.start_run(run_name=run_name) as run:
    tags = {"project":"nostressia","task":"current-stress","features":"all","model":"RF",
            "dataset_name":DATASET_NAME,"dataset_version":DATASET_VERSION,"split":"80/20","random_state":str(RANDOM_STATE)}
    log_run_metadata(tags, description)
    log_dataset_inputs(X_train, y_train, X_test, y_test)
    mlflow.log_params({"model_type":"RandomForestClassifier", "tuning":"GridSearchCV", **{k:str(v) for k,v in search.best_params_.items()}})
    mlflow.log_metric("best_cv_score", float(search.best_score_))
    mlflow.log_metrics(metrics)

    with tempfile.TemporaryDirectory() as td:
        artifact_dir = Path(td)
        log_classification_artifacts(y_test, y_pred, y_proba, labels=sorted(y.unique()), artifact_dir=artifact_dir)
        mlflow.log_artifacts(str(artifact_dir), artifact_path="evaluation")

    log_and_register_model(best_model, X_train, model_name="CurrentStress_RF_Tuned")

print(metrics)
print(f"Run selesai: {run.info.run_id}")
